# Bài tập 2 – Lọc ảnh bằng convolution thủ công

Tính lần lượt **I1 → I6** bằng NumPy, hiển thị và lưu kết quả.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view

IMAGE_PATH = Path("anh1.jpg")
OUTPUT_DIR = Path("output_cau2")

bgr = cv2.imread(str(IMAGE_PATH))
if bgr is None:
    raise FileNotFoundError(f"Không tìm thấy ảnh: {IMAGE_PATH}")

rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
gray = (0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]).astype(np.float32)
print("Kích thước ảnh xám:", gray.shape)

## Các hàm xử lý

In [ ]:
def convolution(image, kernel, padding=0, stride=1):
    padded = np.pad(image, padding, mode="constant")
    windows = sliding_window_view(padded, kernel.shape)[::stride, ::stride]
    return np.sum(windows * kernel, axis=(2, 3)).astype(np.float32)

def median_filter(image, ksize):
    pad = ksize // 2
    padded = np.pad(image, pad, mode="constant")
    windows = sliding_window_view(padded, (ksize, ksize))
    return np.median(windows, axis=(2, 3)).astype(np.float32)

def make_same_size(a, b):
    h, w = max(a.shape[0], b.shape[0]), max(a.shape[1], b.shape[1])
    def center_pad(image):
        out = np.zeros((h, w), dtype=np.float32)
        y, x = (h - image.shape[0]) // 2, (w - image.shape[1]) // 2
        out[y:y + image.shape[0], x:x + image.shape[1]] = image
        return out
    return center_pad(a), center_pad(b)

def normalize(image):
    return cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

## Tính I1 → I6

In [ ]:
kernel3 = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)

kernel5 = np.array([
    [1, 4, 6, 4, 1], [4, 16, 24, 16, 4], [6, 24, 36, 24, 6],
    [4, 16, 24, 16, 4], [1, 4, 6, 4, 1]
], dtype=np.float32)
kernel5 /= kernel5.sum()

kernel7 = np.array([
    [1, 1, 2, 2, 2, 1, 1], [1, 2, 4, 4, 4, 2, 1],
    [2, 4, 8, 8, 8, 4, 2], [2, 4, 8, 16, 8, 4, 2],
    [2, 4, 8, 8, 8, 4, 2], [1, 2, 4, 4, 4, 2, 1],
    [1, 1, 2, 2, 2, 1, 1]
], dtype=np.float32)
kernel7 /= kernel7.sum()

I1 = convolution(gray, kernel3, padding=1)
I2 = convolution(gray, kernel5, padding=2)
I3 = convolution(gray, kernel7, padding=3, stride=2)
I4 = median_filter(I3, 3)
I5 = median_filter(I1, 5)
I4, I5 = make_same_size(I4, I5)
I6 = np.where(I4 > I5, 0, I5)

results = [gray, I1, I2, I3, I4, I5, I6]
titles = ["Ảnh xám", "I1 - Sobel X 3×3", "I2 - Gaussian 5×5",
          "I3 - Gaussian 7×7, stride 2", "I4 - Median 3×3",
          "I5 - Median 5×5 trên I1", "I6 - Kết hợp I4, I5"]

for title, image in zip(titles, results):
    print(f"{title:<30} {image.shape}")

## Hiển thị và lưu kết quả

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, image, title in zip(axes.flat, results, titles):
    ax.imshow(normalize(image), cmap="gray")
    ax.set_title(title)
    ax.axis("off")
axes.flat[-1].axis("off")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
fig.savefig("ket_qua_cau2.png", dpi=100, bbox_inches="tight")
for i, image in enumerate(results[1:], start=1):
    cv2.imwrite(str(OUTPUT_DIR / f"I{i}.jpg"), normalize(image),
                [cv2.IMWRITE_JPEG_QUALITY, 100])
plt.show()
print(f"Đã lưu ảnh tổng hợp: {Path('ket_qua_cau2.png').resolve()}")
print(f"Đã lưu I1-I6 vào: {OUTPUT_DIR.resolve()}")